# Orchestration
orchestration.py is the main file for creating the agentic system. This file's process is:
1. Initialize all assistants (manual)
2. Initialize the orchestration (automatic)
3. Create delegation pattern (automatic)
4. Initialize the group chat

The main focus of this document is to explain what manual writing needs to be done and what the automated processes are doing.

## Imports (and global variable)
The required imports will also have the agents being used, along with any extra parameter they might need.
Here we see that the dbAgent requires the DatabaseConnection as an extra parameter. 
Understanding this is key for seeing what manual writing needs to be done in the `orchestrate()` function.

Since we separated the documentation into a separate folder, we need to change the routing logic for the import statements before anything else.

In [3]:
import sys
from pathlib import Path

def setup_path():
    ai_root = Path().resolve().parent

    if str(ai_root) not in sys.path:
        sys.path.insert(0, str(ai_root))

setup_path()

In [4]:
from autogen import ConversableAgent, LLMConfig,  UserProxyAgent
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group import (
    AgentTarget,
    OnCondition,
    StringLLMCondition,
)
from autogen.agentchat.group.patterns import(
    AutoPattern,
)
from baseAgent import BaseAgent
from adviceAgent import AdviceAgent
from dbAgent import DBAgent, DatabaseConnection
from diagnosisAgent import DiagnosisAgent
from priceAgent import PriceAgent
from statsAgent import StatAgent, CodeExecutor
from utils import load_prompts

LLM_CONFIG = LLMConfig.from_json(path = "OAI_CONFIG_LIST.json")
prompts = load_prompts()

## Initialize Assistant
This function makes the process of initializing assistant agents slightly more simple by automatically adding the config.
The only issue is due to the various input parameters that could be required for an assistant (e.g. dbAgent with `DatabaseConnection()`)

In [5]:
def initAssistant(agent: BaseAgent, key, *args):
    assistant = agent(*args, prompts=prompts[key])
    assistant.agent.llm_config = LLM_CONFIG

    return assistant

## Initialize Orchestrator
This function takes in a list of assistant agents and outputs a fully functional orchestrator.

First, the orchestrator prompt is given. This is not yet hardened and should pose interesting security concerns, specifically with the requirement of handing off and not doing its own reasoning.
We then create the orchestrator with no max consecutive auto replies since we found that the errors to do with auto replying were due to other agents auto replying logic.

Next, for each agent that will be in the system, we register their tools for execution by the orchestrator (detailed further in baseAgent.ipynb), and prepare the delegation pattern based off the description of each agent.
This allows for the orchestrator to understand when to handoff a task to a specific assistant agent.

In [6]:
def initOrchestrator(assistants: [BaseAgent]):
    orchestratorAgent = ConversableAgent(
        name = "orchestrator",
        system_message = prompts["orchestrator"]["instructions"],
        llm_config=LLM_CONFIG,
        human_input_mode="NEVER"
    )
    conditions = []
    
    for assistant in assistants:
        assistant.registerExecution(orchestratorAgent)
        
        conditions.append(
            OnCondition(
                target=AgentTarget(assistant),
                condition=StringLLMCondition(prompt=assistant.description),
            )
        )
    
    orchestratorAgent.handoffs.add_llm_conditions(conditions)
    
    return orchestratorAgent

## Orchestrate
Most of the orchestration logic is done outside of the orchestrate function.
This is due to the web ui calling this function several times with an API, so if we put the logic inside it would take significantly longer.

Notice that this part is the only part that requires manual changes when using more or less agents; everything else is done automatically.

First, we write an initializing statement for each assistant agent.
Due to the nature of some agents requiring additional parameters, this step needs to be done manually.
Focusing on the adviceAgent and dbAgent, we see them getting initialized and passed into the list.
The adviceAgent does not have an additional parameter requirement, so it just calls `initAssistant(AdviceAgent)`.
But the dbAgent requires an additional parameter, so it calls `initAssistant(DBAgent, DatabaseConnection())`.

Once each agent is initialized and put in the assistant array, the remainder of the code does not require changes for a group chat to be held between a dynamic number of agents with tools.

In [11]:
adviceAgent = initAssistant(AdviceAgent, "advice")
dbAgent = initAssistant(DBAgent, "db", DatabaseConnection())
diagnosisAgent = initAssistant(DiagnosisAgent, "diagnosis")
priceAgent = initAssistant(PriceAgent, "price")
statsAgent = initAssistant(StatAgent, "stats", CodeExecutor())

assistants = [adviceAgent, dbAgent, diagnosisAgent, priceAgent, statsAgent]

orchestratorAgent = initOrchestrator(assistants)

[DB] Tables created successfully
[DB] Error seeding data: (sqlite3.InterfaceError) Error binding parameter 1 - probably unsupported type.
[SQL: INSERT INTO medical_history (user_id, conditions, allergies, surgeries) VALUES (?, ?, ?, ?) RETURNING id]
[parameters: (1001, ['Type 2 Diabetes', 'Hypertension'], ['Penicillin', 'Sulfa drugs'], ['Appendectomy (2010)'])]
(Background on this error at: https://sqlalche.me/e/20/rvf5)


We initialize the orchestrator and create a user proxy agent for the user.
We set the auto reply here to zero since there is no reason for any other value to be set, since it is the user's decision to input an auto reply (blank response).

Then we create a pattern that will understand the structure of the system.
This updates dynamically based on the agents in the assistants array.

In [8]:
user = UserProxyAgent(
    name="user",
    human_input_mode="TERMINATE",
    max_consecutive_auto_reply=2,
    code_execution_config={"use_docker": False},
)

agent_pattern = AutoPattern(
    agents=[orchestratorAgent] + [a.agent for a in assistants],
    initial_agent=orchestratorAgent,
    group_manager_args={"llm_config": LLM_CONFIG},
    user_agent=user
)

Lastly, the group chat is initialized.
The max number of rounds is set just for demonstration purposes to 10, since that is the exact number required for one conversation based on the given prompt.

In [9]:
def orchestrate(message: str):
    result, final_context, last_agent = initiate_group_chat(
        pattern=agent_pattern,
        messages=message,
        max_rounds=10,
    )
    return result

## Example
Here is an example usage for the orchestration.
It is just a function call of `orchestrate()` which allows for easy implementation for the web UI.

In [10]:
message = "I am a 55 year old pregnant woman who smokes, can you give me some healthcare advice?"
orchestrate(message)

user (to chat_manager):

I am a 55 year old pregnant woman who smokes, can you give me some healthcare advice?

--------------------------------------------------------------------------------

Next speaker: orchestrator

orchestrator (to chat_manager):


***** Suggested tool call (call_hxac7bip): transfer_to_AdviceAgent_1 *****
Arguments: 
{}
**************************************************************************

--------------------------------------------------------------------------------

Next speaker: _Group_Tool_Executor


>>>>>>>> EXECUTING FUNCTION transfer_to_AdviceAgent_1...
Call ID: call_hxac7bip
Input arguments: {}

>>>>>>>> EXECUTED FUNCTION transfer_to_AdviceAgent_1...
Call ID: call_hxac7bip
Input arguments: {}
Output:
Transfer to AdviceAgent
_Group_Tool_Executor (to chat_manager):

***** Response from calling tool (call_hxac7bip) *****
Transfer to AdviceAgent
******************************************************

---------------------------------------------------

ChatResult(chat_id=114595733816174929860713175303406835219, chat_history=[{'content': 'I am a 55 year old pregnant woman who smokes, can you give me some healthcare advice?', 'role': 'assistant', 'name': 'user'}, {'content': '', 'tool_calls': [{'id': 'call_hxac7bip', 'function': {'arguments': '{}', 'name': 'transfer_to_AdviceAgent_1'}, 'type': 'function', 'index': 0}], 'name': 'orchestrator', 'role': 'assistant'}, {'content': 'Transfer to AdviceAgent', 'tool_responses': [{'tool_call_id': 'call_hxac7bip', 'role': 'tool', 'content': 'Transfer to AdviceAgent'}], 'name': '_Group_Tool_Executor', 'role': 'tool'}, {'content': '', 'tool_calls': [{'id': 'call_3hdk0uwr', 'function': {'arguments': '{"age":"55","pregnant":"yes","sex":"female","tobaccoUse":"yes"}', 'name': 'APITool'}, 'type': 'function', 'index': 0}], 'name': 'AdviceAgent', 'role': 'assistant'}, {'content': '{\'Title\': \'The Basics: Overview\', \'Content\': \'<p>Nearly half of all adults in the United States have high blood pressu